# Stack Overflow Salary Analysis
### Portfolio refresh of a 2019 Thinkful capstone

This notebook is a presentation and reproducibility refresh of an analysis originally completed during Thinkful Data Science training in 2019. The analytical questions, core transformations, salary thresholds, and statistical comparisons are preserved from the original work. The structure, wording, labels, and code organization have been cleaned up for a professional portfolio.

**Original notebook:** `Stackoverflow_users_salary_prediction.ipynb`

The original notebook remains preserved separately. This portfolio version does not claim that the analysis was newly performed in 2026.

## 1. Analytical Objective

The original project explored factors that may be associated with compensation among Stack Overflow survey respondents. This refreshed version organizes that work around four questions:

1. **Age and professional experience:** How does annual compensation vary across age and professional coding experience groups?
2. **Composite segment:** Does the original coursework-defined segment combining education, wake time, and computer hours show a difference in compensation?
3. **Formal education:** How does compensation vary across bachelor's, master's, and doctoral degree groups?
4. **Computer hours:** Among full-time respondents, is there a clear descriptive relationship between time spent on a computer and compensation?

These are observational survey data. The analysis can identify patterns and statistical associations, but it cannot establish that any factor causes a change in salary.

## 2. Data Source and Scope

The project uses the Stack Overflow Developer Survey dataset used in the original Thinkful capstone.

The analysis focuses on respondents reporting compensation in U.S. dollars and uses the following fields:

- Age
- Gender
- Employment status
- Salary and salary reporting period
- Professional coding experience
- Formal education
- Wake time
- Hours spent on a computer

### Important limitations

The survey is self-reported and self-selected, so the sample should not be treated as representative of every technology worker. Salary values also require normalization because respondents reported compensation using different time periods. The original project used heuristic salary bounds to remove extreme or implausible values; those same thresholds are retained here for consistency.

## 3. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

%matplotlib inline
plt.rcParams["font.size"] = 12

## 4. Load the Data

The 2019 notebook referenced a local Windows path (`F:/Thinkful/Files/...`), which made it difficult for another person to rerun. This version looks for the CSV in a repository-level `data/` folder or in a nearby `data/` folder.

Place `survey_results_public.csv` in `data/` before running the notebook.

In [ ]:
DATA_CANDIDATES = [
    Path("data/survey_results_public.csv"),
    Path("../../../../data/survey_results_public.csv"),
]

DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "survey_results_public.csv was not found. "
        "Place the file in the repository's data/ folder and rerun this cell."
    )

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} survey responses from {DATA_PATH}")

## 5. Data Preparation

The preparation below follows the original workflow:

1. Select the columns used in the analysis.
2. Restrict the sample to respondents reporting salary in USD.
3. Simplify formal education labels.
4. Convert salary strings to numeric values.
5. Normalize weekly and monthly salaries to annual amounts.
6. Retain annual salary values between **$2,000 and $500,000**, matching the original project.

The salary range is a heuristic from the original coursework, not a universally valid definition of an outlier.

In [ ]:
analysis_columns = [
    "Age",
    "Gender",
    "Employment",
    "Salary",
    "SalaryType",
    "CurrencySymbol",
    "YearsCodingProf",
    "FormalEducation",
    "WakeTime",
    "HoursComputer",
]

df_new = df[analysis_columns].copy()
df_new = df_new[df_new["CurrencySymbol"] == "USD"].copy()

education_labels = {
    "Some college/university study without earning a degree": "C W/O D",
    "Bachelor’s degree (BA, BS, B.Eng., etc.)": "BA",
    "Master’s degree (MA, MS, M.Eng., MBA, etc.)": "MA",
    "Associate degree": "AA",
    "Other doctoral degree (Ph.D, Ed.D., etc.)": "Ph.D",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "SecondarySchool",
    "Professional degree (JD, MD, etc.)": "ProDegree",
    "Primary/elementary school": "PrimarySchool",
    "I never completed any formal education": "NoEducation",
}

df_new["FormalEducation"] = df_new["FormalEducation"].map(education_labels)

df_new["Salary"] = (
    df_new["Salary"]
    .astype(str)
    .str.replace(",", "", regex=False)
)
df_new["Salary"] = pd.to_numeric(df_new["Salary"], errors="coerce")

# The original notebook dropped rows with missing values across the selected fields.
df_new = df_new.dropna().copy()

salary_multiplier = {
    "Yearly": 1,
    "Monthly": 12,
    "Weekly": 52,
}

df_new["Converted_Salary"] = (
    df_new["Salary"] * df_new["SalaryType"].map(salary_multiplier)
)

# Preserve the original capstone's salary bounds.
df_new = df_new[
    (df_new["Converted_Salary"] > 2_000)
    & (df_new["Converted_Salary"] < 500_000)
].copy()

print(f"Analysis sample: {len(df_new):,} respondents")
df_new["Converted_Salary"].describe()

### Preparation check

A quick distribution check helps confirm that the cleaned salary field is usable before moving into the analytical questions.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_new["Converted_Salary"], bins=40)
ax.set_title("Distribution of Annualized Salary")
ax.set_xlabel("Annualized salary (USD)")
ax.set_ylabel("Respondents")
plt.show()

## 6. Question 1 — Age, Professional Experience, and Compensation

The original capstone treated age and years of professional coding experience as primary descriptive factors. This refreshed section keeps that question but uses **median salary** for the visual summaries because salary distributions are typically skewed and the median is less sensitive to high earners.

The professional-experience categories are kept in the same order used by the original project.

In [ ]:
age_order = [
    "Under 18 years old",
    "18 - 24 years old",
    "25 - 34 years old",
    "35 - 44 years old",
    "45 - 54 years old",
    "55 - 64 years old",
    "65 years or older",
]

experience_order = [
    "0-2 years",
    "3-5 years",
    "6-8 years",
    "9-11 years",
    "12-14 years",
    "15-17 years",
    "18-20 years",
    "21-23 years",
    "24-26 years",
    "27-29 years",
    "30 or more years",
]

observed_age_order = [x for x in age_order if x in df_new["Age"].unique()]
observed_experience_order = [
    x for x in experience_order if x in df_new["YearsCodingProf"].unique()
]

median_by_age = (
    df_new.groupby("Age")["Converted_Salary"]
    .median()
    .reindex(observed_age_order)
    .dropna()
)

median_by_experience = (
    df_new.groupby("YearsCodingProf")["Converted_Salary"]
    .median()
    .reindex(observed_experience_order)
    .dropna()
)

fig, ax = plt.subplots(figsize=(9, 4))
median_by_age.plot(kind="bar", ax=ax)
ax.set_title("Median Annualized Salary by Age Group")
ax.set_xlabel("Age group")
ax.set_ylabel("Median salary (USD)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
median_by_experience.plot(kind="bar", ax=ax)
ax.set_title("Median Annualized Salary by Professional Coding Experience")
ax.set_xlabel("Professional coding experience")
ax.set_ylabel("Median salary (USD)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Interpretation

In the original analysis, salary generally increased across older age groups and longer professional coding experience groups. This is a **descriptive association**, not evidence that age itself causes higher pay. Age and professional experience are also related to one another, so a stronger follow-up analysis would model them jointly rather than interpreting either factor in isolation.

## 7. Question 2 — Coursework-Defined Composite Segment

The original capstone created a custom segment from three variables:

- completed higher education,
- an earlier wake time,
- nine or more hours of computer use.

The 2019 notebook called the combined group **“Good”** and everyone else **“ILivemyway.”** Those labels are subjective, so this portfolio copy uses neutral labels while preserving the exact underlying logic.

This segment was an exploratory coursework construction, **not a validated measure of lifestyle or productivity**.

In [ ]:
higher_education = {
    "BA",
    "MA",
    "Ph.D",
    "AA",
    "ProDegree",
}

df_new["EducationSegment"] = np.where(
    df_new["FormalEducation"].isin(higher_education),
    "HigherEducationGroup",
    "OtherEducationGroup",
)

early_wake_times = {
    "Before 5:00 AM",
    "Between 5:00 - 6:00 AM",
    "Between 6:01 - 7:00 AM",
    "Between 7:01 - 8:00 AM",
}

df_new["WakeSegment"] = np.where(
    df_new["WakeTime"].isin(early_wake_times),
    "EarlyWake",
    "OtherWakeTime",
)

long_computer_hours = {
    "9 - 12 hours",
    "Over 12 hours",
}

df_new["ComputerSegment"] = np.where(
    df_new["HoursComputer"].isin(long_computer_hours),
    "9PlusHours",
    "Under9Hours",
)

df_new["CompositeSegment"] = np.where(
    (df_new["EducationSegment"] == "HigherEducationGroup")
    & (df_new["WakeSegment"] == "EarlyWake")
    & (df_new["ComputerSegment"] == "9PlusHours"),
    "DefinedSegment",
    "OtherRespondents",
)

segment_summary = (
    df_new.groupby("CompositeSegment")["Converted_Salary"]
    .agg(["count", "mean", "median", "std"])
)
segment_summary

In [ ]:
defined_segment = df_new.loc[
    df_new["CompositeSegment"] == "DefinedSegment",
    "Converted_Salary",
]

other_segment = df_new.loc[
    df_new["CompositeSegment"] == "OtherRespondents",
    "Converted_Salary",
]

segment_test = ttest_ind(
    defined_segment,
    other_segment,
    equal_var=False,
    nan_policy="omit",
)

print("Welch t-test: DefinedSegment vs OtherRespondents")
print(f"t-statistic: {segment_test.statistic:.3f}")
print(f"p-value: {segment_test.pvalue:.4g}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_new.boxplot(
    column="Converted_Salary",
    by="CompositeSegment",
    ax=ax,
    grid=False,
)
ax.set_title("Annualized Salary by Coursework-Defined Composite Segment")
ax.set_xlabel("")
ax.set_ylabel("Annualized salary (USD)")
plt.suptitle("")
plt.show()

### Interpretation

The original notebook did not find strong statistical support for treating this constructed segment as a meaningful salary factor. More importantly, the segment combines several different concepts into a single binary label, and **education is embedded inside the definition itself**. That makes it difficult to interpret any salary difference as a distinct “lifestyle” effect.

For a professional analysis, this segment is best treated as an exploratory feature-engineering exercise rather than a substantive conclusion. Its limitations motivate examining education directly in the next section.

## 8. Question 3 — Formal Education and Compensation

The original project compared respondents whose highest formal education was a bachelor's degree, master's degree, or doctoral degree.

This section preserves the pairwise **Welch independent-samples t-tests** used in the 2019 notebook. Welch's test is preferable to the equal-variance form when group variances may differ.

In [ ]:
degree_groups = ["BA", "MA", "Ph.D"]

degree_data = {
    degree: df_new.loc[
        df_new["FormalEducation"] == degree,
        "Converted_Salary",
    ]
    for degree in degree_groups
}

degree_summary = pd.DataFrame({
    degree: {
        "n": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "std": values.std(),
    }
    for degree, values in degree_data.items()
}).T

degree_summary

In [ ]:
comparisons = [
    ("BA", "MA"),
    ("Ph.D", "MA"),
    ("Ph.D", "BA"),
]

test_results = []

for left, right in comparisons:
    result = ttest_ind(
        degree_data[left],
        degree_data[right],
        equal_var=False,
        nan_policy="omit",
    )
    test_results.append({
        "Comparison": f"{left} vs {right}",
        "t_statistic": result.statistic,
        "p_value": result.pvalue,
    })

pd.DataFrame(test_results)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df_new[df_new["FormalEducation"].isin(degree_groups)].boxplot(
    column="Converted_Salary",
    by="FormalEducation",
    ax=ax,
    grid=False,
)
ax.set_title("Annualized Salary by Formal Education")
ax.set_xlabel("Highest formal education")
ax.set_ylabel("Annualized salary (USD)")
plt.suptitle("")
plt.show()

### Interpretation

In the original 2019 output, the pairwise tests reported approximate p-values of:

- **BA vs. MA:** 0.0004
- **BA vs. Ph.D:** 0.015
- **MA vs. Ph.D:** 0.31

At a conventional 0.05 threshold, the original results suggested differences for BA vs. MA and BA vs. Ph.D, but not for MA vs. Ph.D.

These comparisons should be interpreted cautiously. They are unadjusted pairwise tests from observational survey data and do not control for professional experience, job type, location within the United States, specialty, or other potential confounders. They therefore support an **association** between education group and observed compensation in this sample, not a causal claim that obtaining a particular degree increases salary.

## 9. Exploratory Check — Computer Hours and Compensation

The original notebook also examined computer hours among full-time respondents. The 2019 interpretation stated that more computer time did not produce a salary advantage. A scatter plot alone cannot prove that claim, so the refreshed interpretation is intentionally more limited.

In [ ]:
full_time = df_new[df_new["Employment"] == "Employed full-time"].copy()

hour_order = [
    "Less than 1 hour",
    "1 - 4 hours",
    "5 - 8 hours",
    "9 - 12 hours",
    "Over 12 hours",
]

median_by_hours = (
    full_time.groupby("HoursComputer")["Converted_Salary"]
    .median()
    .reindex([x for x in hour_order if x in full_time["HoursComputer"].unique()])
    .dropna()
)

fig, ax = plt.subplots(figsize=(8, 4))
median_by_hours.plot(kind="bar", ax=ax)
ax.set_title("Median Salary by Computer Hours — Full-Time Respondents")
ax.set_xlabel("Reported computer hours")
ax.set_ylabel("Median annualized salary (USD)")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

### Interpretation

For full-time respondents, the descriptive pattern does not establish a simple “more hours = more salary” relationship. Computer hours may reflect many different things, including role type, work style, or non-work computer use. This exploratory check does not validate the composite segment from Question 2 and should not be interpreted causally.

## 10. Key Findings

The refreshed analysis preserves the main takeaways from the original capstone while using more cautious statistical language:

- **Professional experience and salary:** Compensation generally increased across longer professional coding experience groups in the survey sample.
- **Age and salary:** Older age groups also tended to show higher compensation, although age and experience are related and should not be interpreted independently as causal drivers.
- **Composite segment:** The original education/wake-time/computer-hours segment did not provide a strong basis for claiming a distinct lifestyle effect on salary.
- **Formal education:** Bachelor's, master's, and doctoral groups showed different compensation distributions. The original pairwise Welch tests found statistically significant differences for BA vs. MA and BA vs. Ph.D, but not MA vs. Ph.D.
- **Computer hours:** Among full-time respondents, computer hours did not show a simple descriptive relationship with compensation.

The strongest portfolio value of this project is the workflow itself: selecting variables from a large survey, cleaning inconsistent salary data, engineering analytical groups, visualizing distributions, applying statistical tests, and revising conclusions based on limitations.

## 11. Limitations

This project has several important limitations:

1. **Self-selected survey sample:** Stack Overflow respondents may differ from the broader technology workforce.
2. **Self-reported compensation:** Salary and work-pattern variables may contain reporting error.
3. **Heuristic outlier filtering:** The $2,000–$500,000 annual salary range is preserved from the original coursework and is not a formally derived outlier rule.
4. **Potential confounding:** Age, professional experience, education, job type, specialty, and other variables may be related.
5. **Multiple pairwise tests:** The education comparisons were not adjusted for multiple testing.
6. **Composite-segment design:** The coursework-defined segment is subjective and combines several variables, including education, making it difficult to interpret.
7. **No multivariable model in this analysis:** The project does not isolate independent effects of individual predictors.

These limitations are important because statistical significance alone does not establish practical importance or causality.

## 12. Possible Next Steps

If this analysis were extended today, useful next steps would include:

- build a multivariable regression model to estimate associations while controlling for experience and other covariates;
- use more systematic outlier handling and sensitivity analyses;
- report confidence intervals and effect sizes alongside p-values;
- adjust for multiple pairwise comparisons;
- separate work-related and non-work computer time if the data support it;
- evaluate model assumptions and predictive performance on held-out data.

Those extensions are **recommendations**, not work claimed as completed in the original 2019 project.

---

### Portfolio provenance

**Original work:** Thinkful Data Science capstone, 2019  
**Portfolio refresh:** organization, documentation, reproducibility, neutral labeling, and statistical interpretation updated for presentation.  
**Original notebook preserved:** `Stackoverflow_users_salary_prediction.ipynb`